# **Problem Statement**

## **Business Context**

ShopNest Global is a large-scale e-commerce platform operating across 30+ countries, serving over 50 million active customers and employing 2,000+ human support agents who run 24/7 across the US, Europe, India, and Southeast Asia, across product categories including electronics, fashion, groceries, and home appliances.

ShopNest processes over 200,000 orders per day. With every order comes the possibility of a delivery delay, a payment failure, a wrong item, or a return request. Customers reach out to the support team through tickets to report these issues and expect a fast, accurate resolution. These tickets are written in highly unstructured ways:

- Some are overloaded with background details where the real issue is buried.
- Others use abbreviations, shorthand, and order codes that are hard to interpret.
- Some contain so little information that the issue is entirely unclear.

As a result:

- Human agents spend the first **2–3 minutes** on each ticket just decoding what the customer is asking before any resolution work can begin.
- During peak periods (sales, holidays, logistics disruptions), daily ticket volume can spike from ~5,000 to **~15,000**, multiplying this inefficiency.
- High volume and inconsistent ticket content contribute to agent fatigue and higher error rates when accuracy is most critical.
- Drafting responses is manual, slow, and inconsistent in tone and clarity across agents, leading to suboptimal customer experiences.

## **Objective**

The objective is to build a POC of an AI-powered ticket intelligence system for ShopNest Global that:

1. **Summarises** incoming raw, unstructured tickets into a clean, concise summary for the support agent.
2. **Evaluates** the generated summary using an LLM-as-Judge approach, scoring quality on defined criteria.
3. **Generates** a professional, empathetic customer response grounded in ShopNest's support policies.
4. **Evaluates** the generated response using an LLM-as-Judge approach, scoring resolution quality.
5. **Compiles** all outputs into a single structured table and exports it for downstream use.

The end goal is to demonstrate that AI-assisted summarisation and response generation can meaningfully improve the consistency and quality of customer support operations at scale.

## **Data Dictionary**

| Column Name         | Data Type | Description                                                       |
| ------------------- | --------- | ----------------------------------------------------------------- |
| support_ticket_id   | Integer   | Unique identifier assigned to each support ticket                 |
| support_ticket_text | String    | Free-form text describing the issue or request raised by the user |

# **Installing and Importing Necessary Libraries**

In [ ]:
# Install LangChain and OpenAI for LLM API access, and pandas for data handling
# Pinned versions ensure reproducibility across environments
%pip install pandas==2.2.2 langchain-openai==1.1.12 openai==2.31.0

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for VSCode), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [ ]:
pip install pandas

In [ ]:
%pip install -U langchain-openai openai

In [1]:
# Core libraries
import os            # For environment variable access (API key fallback)
import json
import re            # For parsing structured evaluator output into numeric scores
import pandas as pd  # For loading, manipulating, and exporting tabular data
from langchain_openai import ChatOpenAI  # OpenAI
from openai import OpenAI

## **Loading the Open API Key**

In [2]:
# Load the JSON file and extract values
file_name = 'C:\\csagents\\agenticAIFoundations\\Agentic-AI-Foundation\\config.json'                                                      # Name of the configuration file
with open(file_name, 'r') as file:                                              # Open the config file in read mode
    config = json.load(file)                                                    # Load the JSON content as a dictionary
    OPENAI_API_KEY = config.get("OPENAI_API_KEY")                                             # Extract the API key from the config
    OPENAI_API_BASE = config.get("OPENAI_API_BASE")
# Store API credentials in environment variables
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY                                  # Set API key as environment variable
os.environ["OPENAI_BASE_URL"] = OPENAI_API_BASE                                 # Set API base URL as environment variable
client = OpenAI()

# **Data Loading**

## **Load the data**

In [3]:
file_path = r"C:\csagents\agenticAIFoundations\Agentic-AI-Foundation\support_ticket_data.csv"
df = pd.read_csv(file_path)

In [4]:
df.shape

(30, 2)

## **Data Overview**

In [5]:
print(df.head())

   support_ticket_id                                support_ticket_desc
0                  1  I cannot believe the level of service I have r...
1                  2  Ord SNX-8902 ACH debit failed at checkout, tri...
2                  3                          not working. please help.
3                  4  Okay so I have genuinely had it with this comp...
4                  5                                refund not received


In [6]:
print(df.tail())

    support_ticket_id                                support_ticket_desc
25                 26  Hi, just wanted to check on the status of my o...
26                 27  Hello, quick question - what is your standard ...
27                 28  Hi there. I received my order SNX-7823 today a...
28                 29  Hey, I placed an order yesterday evening - ord...
29                 30  Hi, I was just checking whether my recent retu...


# **Summarization**

## **Set up an LLM**

I selected `gpt-4o-mini` for summarization because summaries are short, factual, and generated at high volume. Its lower cost and latency suit routine ticket processing while the constrained prompt preserves reliability.

In [7]:
SUMMARIZATION_MODEL = "gpt-4o-mini"

## **Set parameters**

The low temperature reduces creative variation and keeps summaries grounded. The 50-token limit is enough for one or two sentences and controls cost. A top-p value of 0.95 allows natural wording without broad, unpredictable sampling.

In [8]:
SUMMARY_TEMP = 0.1
SUMMARY_MAX_TOKENS = 50
SUMMARY_TOP_P = 0.95

## **Prompting Technique**

I use separate system and user messages. The system message defines the summarization role and constraints, while the user message supplies the ticket. This makes the prompt reusable and reduces unsupported additions.

### **System Message**

In [9]:
SUMMARISER_SYSTEM = """
You are an assistant that converts raw customer support tickets into a concise, agent-ready summary.
Extract the main issue, relevant details, and the customer’s request.
Keep the summary short, factual, and free of extra explanation.
Do not add anything that is not present in the ticket.
"""

In [10]:
SUMMARISER_USER = """
Read the ticket below and write a brief support summary in one or two sentences.
Ticket: "{ticket}"
Summary:
"""

### **Generate Ticket Summaries**

In [11]:
def generate_summary(ticket: str) -> str:
    user_message = SUMMARISER_USER.format(ticket=ticket)

    response = client.chat.completions.create(
        model=SUMMARIZATION_MODEL,
        temperature=SUMMARY_TEMP,
        max_tokens=SUMMARY_MAX_TOKENS,
        top_p=SUMMARY_TOP_P,
        messages=[
            {"role": "system", "content": SUMMARISER_SYSTEM},
            {"role": "user", "content": user_message}
        ]
    )

    return response.choices[0].message.content.strip()

In [12]:
test_ticket = df["support_ticket_desc"].iloc[0]

print("INPUT TICKET:")
print("-" * 60)
print(test_ticket)

print("\nGENERATED SUMMARY:")
print("-" * 60)
test_summary = generate_summary(test_ticket)
print(test_summary)

INPUT TICKET:
------------------------------------------------------------
I cannot believe the level of service I have received. I have been a loyal ShopNest customer since 2020 and have spent thousands of dollars on this platform. I have recommended ShopNest to everyone I know - my coworkers, my neighbors, my entire book club. I have never once filed a complaint or asked for anything special. I always leave five star reviews and I tip delivery drivers generously. And THIS is how you treat your most loyal customers? I am beyond frustrated. I spent over three hours today trying to reach your support team and not a single person gave me a straight answer. Your chatbot is completely useless, your email support is nonexistent, and your phone line had a 40 minute hold time. I have screenshots of every single interaction. I will be filing a complaint with the Better Business Bureau if this is not resolved by end of day. Anyway. Order SNX-4421 for a Bosch dishwasher. Delivered wrong model. W

### **Observations**

As a learner, I observed that increasing temperature can make the summary more varied, but it may also introduce details that are not supported by the ticket. For a reliable support workflow, I would keep temperature low so the summary stays grounded in the source.

I also observed that max tokens controls the output limit rather than guaranteeing a better summary. The limit should be large enough to include the important facts, but not so large that the model produces unnecessary content.

# **Evaluation for Summarization**

## **Setup an LLM**

I selected `gpt-4o-mini` for summary evaluation because the rubric is explicit and the output is a small structured scorecard. This keeps high-volume evaluation cost-effective while invalid or uncertain results can be routed for review.

In [23]:
# Cost-conscious evaluator model; strong instruction following for structured grading.
SUMMARY_EVAL_MODEL = "gpt-4o-mini"

## **System Message**

In [24]:
SUMMARY_EVAL_SYSTEM = """
You are a JSON evaluator for customer support ticket summaries.
Compare the summary against the original ticket and return ONLY valid JSON.
Include keys: scores, overall_verdict.
The scores object must contain technical_accuracy, completeness, conciseness, hallucination_check.
"""

In [25]:
SUMMARY_EVAL_USER = """\
Original Ticket: "{ticket}"
Ticket Summary: "{summary}"
Evaluate the summary and respond ONLY with valid JSON like:
{{
  "scores": {{
    "technical_accuracy": 1,
    "completeness": 1,
    "conciseness": 1,
    "hallucination_check": 1
  }},
  "overall_verdict": "..."
}}
"""

## **Generate Evaluation Scores**

In [26]:
def evaluate_summary(ticket: str, summary: str) -> str:
    # Escape any literal braces in ticket or summary text before formatting the prompt
    safe_ticket = ticket.replace("{", "{{").replace("}", "}}")
    safe_summary = summary.replace("{", "{{").replace("}", "}}")
    user_message = SUMMARY_EVAL_USER.format(ticket=safe_ticket, summary=safe_summary)

    response = client.chat.completions.create(
        model=SUMMARY_EVAL_MODEL,
        messages=[
            {"role": "system", "content": SUMMARY_EVAL_SYSTEM},
            {"role": "user",   "content": user_message}
        ]
    )
    return response.choices[0].message.content.strip()

In [27]:
print("SUMMARISATION EVALUATION")
print("=" * 60)

test_sum_eval_raw = evaluate_summary(test_ticket, test_summary)
print(test_sum_eval_raw)

SUMMARISATION EVALUATION
{
  "scores": {
    "technical_accuracy": 1,
    "completeness": 0.8,
    "conciseness": 1,
    "hallucination_check": 1
  },
  "overall_verdict": "The summary is technically accurate and concise but lacks completeness as it does not capture all key customer sentiments, like their intention to file a complaint with the Better Business Bureau."
}


## **Observations**

As a learner, I observed that the evaluator scores depend on both the model and the generation parameters. A good evaluation should therefore compare models under consistent prompts and settings. I also learned that structured JSON output makes the scores easier to parse and use in an automated agentic workflow.

# **Response Generation**

## **Set up an LLM**

I selected `gpt-4o-mini` for response drafting because the task requires concise, empathetic language grounded in the summary. It is cost-effective for routine drafting, while policy-sensitive cases can be escalated to a human.

In [44]:
# Separate evaluator model with stronger reasoning for response/policy grading.
RESPONSE_GENERATION_MODEL = "gpt-4o-mini"

## **Set parameters**

A temperature of 0.5 balances consistent support language with natural empathy. The system prompt and policy rules constrain the model so the moderate temperature does not lead to unsupported promises.

In [45]:
GENERATION_TEMP = 0.5

## **Prompting Technique**

This uses a two-stage agentic pattern: one step creates a compact case representation and the next uses that representation to draft the response. The handoff reduces noise and makes each step easier to evaluate.

### **System Message**

In [46]:
GENERATOR_SYSTEM = """
You are a customer support assistant. Use the ticket summary to write a polite, helpful reply.
Respond directly to the issue, offer a clear next step or resolution, and keep the tone professional and empathetic.
Follow these policies: do not claim that a refund, replacement, credit, escalation, shipment, or account change has already been completed unless the summary confirms it; do not invent order details, timelines, contact information, or policy exceptions; acknowledge the customer's concern; request missing information when needed; and recommend human escalation for policy-sensitive or unresolved cases.
Do not add any information that is not supported by the summary.
"""

In [47]:
GENERATOR_USER = """
Use the summary below to write a customer-facing response.
Keep the reply concise, courteous, and focused on resolving the ticket.
Ticket Summary: "{summary}"
"""

### **Generate a User Response**

In [48]:
def generate_response(summary: str) -> str:
    # Inject the summary into the CoT user message template
    user_message = GENERATOR_USER.format(summary=summary)

    response = client.chat.completions.create(
        model=RESPONSE_GENERATION_MODEL,
        temperature=GENERATION_TEMP,       # Moderate temperature for natural, empathetic tone
        messages=[
            {"role": "system", "content": GENERATOR_SYSTEM},
            {"role": "user",   "content": user_message}
        ]
    )
    return response.choices[0].message.content.strip()

In [49]:
print("INPUT SUMMARY:")
print("-" * 60)
print(test_summary)

print("\nGENERATED RESPONSE:")
print("-" * 60)
test_response = generate_response(test_summary)
print(test_response)

INPUT SUMMARY:
------------------------------------------------------------
Customer is frustrated with the service received after being a loyal ShopNest customer since 2020. They spent over three hours trying to reach support without success and are requesting a replacement for order SNX-4421, which was delivered with the wrong model

GENERATED RESPONSE:
------------------------------------------------------------
Dear [Customer's Name],

Thank you for reaching out and for being a loyal ShopNest customer since 2020. I sincerely apologize for the inconvenience you experienced while trying to contact our support team and for the issue with your recent order SNX-4421.

To resolve this matter promptly, I have initiated the process to send you the correct model as a replacement. You will receive a confirmation email shortly with the details and expected delivery date.

If you have any further questions or need additional assistance, please feel free to reply to this message. We truly appre

### **Observations**

As a learner, I observed that response generation is different from summarization: the model must show empathy, acknowledge the customer’s issue, and propose an appropriate next step without inventing actions. The summary acts as a compact state passed from one agentic step to the next, so its accuracy directly affects the final response.

# **Evaluation for Response Generation**

## **Setup an LLM**

I selected `gpt-4o-mini` for response evaluation because the criteria are explicit and the output is a compact structured scorecard. This supports cost-effective batch evaluation while failed or policy-sensitive cases remain eligible for human review.

In [50]:
RESPONSE_EVAL_MODEL = "gpt-4o-mini"

## **System Message**

In [51]:
RESPONSE_EVAL_SYSTEM = """
You are a JSON evaluator for customer support responses.
Compare the generated response against the ticket summary and return ONLY valid JSON.
Include keys: scores, feedback, overall_verdict.
The scores object must contain alignment_with_summary, actionability, tone_empathy, policy_compliance.
The feedback object must contain strengths, weaknesses, risk_factors.
"""

In [52]:
RESPONSE_EVAL_USER = """\
Ticket Summary: "{summary}"
Generated Response: "{response}"

Evaluate the response and respond ONLY with valid JSON like:
{{
  "scores": {{
    "alignment_with_summary": 1,
    "actionability": 1,
    "tone_empathy": 1,
    "policy_compliance": 1
  }},
  "feedback": {{
    "strengths": "...",
    "weaknesses": "...",
    "risk_factors": "..."
  }},
  "overall_verdict": "..."
}}

"""

## **Generate Evaluation Scores**

In [53]:
def evaluate_response(summary: str, response_text: str) -> str:
    # Escape any literal braces in summary or response text before formatting the prompt
    safe_summary = summary.replace("{", "{{").replace("}", "}}")
    safe_response = response_text.replace("{", "{{").replace("}", "}}")
    user_message = RESPONSE_EVAL_USER.format(summary=safe_summary, response=safe_response)

    response = client.chat.completions.create(
        model=RESPONSE_EVAL_MODEL,
        messages=[
            {"role": "system", "content": RESPONSE_EVAL_SYSTEM},
            {"role": "user",   "content": user_message}
        ]
    )
    return response.choices[0].message.content.strip()

In [54]:
print("RESPONSE EVALUATION")
print("=" * 60)

test_resp_eval_raw = evaluate_response(test_summary, test_response)
print(test_resp_eval_raw)

RESPONSE EVALUATION
{
  "scores": {
    "alignment_with_summary": 1,
    "actionability": 1,
    "tone_empathy": 1,
    "policy_compliance": 1
  },
  "feedback": {
    "strengths": "The response effectively acknowledges the customer's loyalty and frustration, shows empathy, and provides a clear action plan to resolve the issue.",
    "weaknesses": "The response could include more specific information about how their support system will improve to prevent future issues.",
    "risk_factors": "If the replacement process takes longer than expected, it may frustrate the customer further."
  },
  "overall_verdict": "The response is well-aligned with the customer's issue, empathetic, actionable, and compliant with typical customer support policies."
}


## **Observations**

As a learner, I observed that response evaluation checks more than grammatical quality. It measures alignment with the summary, actionability, empathy, and policy compliance. This demonstrates how an evaluator can act as a quality-control agent, identifying unsupported promises or missing actions before a response is delivered.

# **Output & Compilation**

In [55]:
# Write your code here
# Initialise storage lists - one entry per ticket, in the same order as the DataFrame
summaries = []   # Generated summaries
sum_eval  = []   # Parsed summarisation evaluation dicts
responses = []   # Generated customer responses
resp_eval = []   # Parsed response evaluation dicts

In [56]:
for idx, row in df.iterrows():
    ticket_id   = row["support_ticket_id"]
    description = row["support_ticket_desc"]

    # ── Step 1: Generate summary ──────────────────────────────────────────────
    summary = generate_summary(description)
    summaries.append(summary)

    # ── Step 2: Evaluate the summary ─────────────────────────────────────────
    sum_eval.append(evaluate_summary(description, summary))

    # ── Step 3: Generate customer response ───────────────────────────────────
    response = generate_response(summary)
    responses.append(response)

    # ── Step 4: Evaluate the response ────────────────────────────────────────
    resp_eval.append(evaluate_response(summary, response))

    print(f"  Ticket {ticket_id} - done")

print(f"\nPipeline complete. Processed {len(df)} tickets.")
print(f"Summaries: {len(summaries)}, Responses: {len(responses)}, Summary evals: {len(sum_eval)}, Response evals: {len(resp_eval)}")

  Ticket 1 - done
  Ticket 2 - done
  Ticket 3 - done
  Ticket 4 - done
  Ticket 5 - done
  Ticket 6 - done
  Ticket 7 - done
  Ticket 8 - done
  Ticket 9 - done
  Ticket 10 - done
  Ticket 11 - done
  Ticket 12 - done
  Ticket 13 - done
  Ticket 14 - done
  Ticket 15 - done
  Ticket 16 - done
  Ticket 17 - done
  Ticket 18 - done
  Ticket 19 - done
  Ticket 20 - done
  Ticket 21 - done
  Ticket 22 - done
  Ticket 23 - done
  Ticket 24 - done
  Ticket 25 - done
  Ticket 26 - done
  Ticket 27 - done
  Ticket 28 - done
  Ticket 29 - done
  Ticket 30 - done

Pipeline complete. Processed 30 tickets.
Summaries: 30, Responses: 30, Summary evals: 30, Response evals: 30


## **Combine All Outputs into a Single Consolidated Table**

In [57]:
# Write your code here
output_df = pd.DataFrame({
    "support_ticket_id"  : df["support_ticket_id"].values,
    "support_ticket_desc": df["support_ticket_desc"].values,
    "generated_summary"  : summaries,
    "generated_response" : responses
})

In [58]:
display(output_df.head(5))
print("Output dataframe shape:", output_df.shape)

,support_ticket_id,support_ticket_desc,generated_summary,generated_response
0,1,I cannot believe the level of service I have r...,Customer is frustrated with support service an...,"Dear [Customer's Name],\n\nThank you for reach..."
1,2,"Ord SNX-8902 ACH debit failed at checkout, tri...",Customer's order SNX-8902 failed due to ACH de...,"Dear [Customer's Name],\n\nThank you for reach..."
2,3,not working. please help.,The customer reports an unspecified issue and ...,"Dear Customer,\n\nThank you for reaching out t..."
3,4,Okay so I have genuinely had it with this comp...,Customer is frustrated with repeated order iss...,"Dear [Customer's Name],\n\nThank you for reach..."
4,5,refund not received,Customer reports not receiving a refund.,"Dear [Customer's Name],\n\nThank you for reach..."


Output dataframe shape: (30, 4)


In [60]:
output_path = "C:\\csagents\\agenticAIFoundations\\Agentic-AI-Foundation\\output.csv"  
output_df.to_csv(output_path, index=False)                       # Save the file

print(f"Output saved to       : {output_path}")

Output saved to       : C:\csagents\agenticAIFoundations\Agentic-AI-Foundation\output.csv


### Save the Evaluation Scores for Summarization and Response Generation

In [65]:
def parse_evaluator_json(eval_item):
    """Parse evaluator JSON and retain diagnostics instead of silently dropping failures."""
    raw = "" if eval_item is None else str(eval_item)
    cleaned = raw.strip()
    if cleaned.startswith("```"):
        cleaned = re.sub(r"^```(?:json)?\s*|\s*```$", "", cleaned, flags=re.IGNORECASE | re.DOTALL).strip()
    try:
        return json.loads(cleaned), raw, ""
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", cleaned, flags=re.DOTALL)
        if match:
            try:
                return json.loads(match.group(0)), raw, "Parsed JSON object from surrounding text"
            except json.JSONDecodeError:
                pass
        return {}, raw, "Invalid JSON returned by evaluator"

summary_evaluations = []
for eval_item in sum_eval:
    eval_data, raw_eval, parse_error = parse_evaluator_json(eval_item)

    scores = eval_data.get("scores", {})
    overall_verdict = eval_data.get("overall_verdict")

    evaluation_entry = {
        "technical_accuracy": scores.get("technical_accuracy"),
        "completeness": scores.get("completeness"),
        "conciseness": scores.get("conciseness"),
        "hallucination_check": scores.get("hallucination_check"),
        "summary_overall_verdict": overall_verdict,
        "evaluation_parse_status": "ok" if not parse_error else parse_error,
        "raw_evaluation_output": raw_eval,
    }
    summary_evaluations.append(evaluation_entry)

summary_eval_df = pd.DataFrame(summary_evaluations)

response_evaluations = []
for eval_item in resp_eval:
    eval_data, raw_eval, parse_error = parse_evaluator_json(eval_item)

    scores = eval_data.get("scores", {})
    feedback = eval_data.get("feedback", {})
    overall_verdict = eval_data.get("overall_verdict")

    evaluation_entry = {
        "alignment_with_summary": scores.get("alignment_with_summary"),
        "actionability": scores.get("actionability"),
        "tone_empathy": scores.get("tone_empathy"),
        "policy_compliance": scores.get("policy_compliance"),
        "response_overall_verdict": overall_verdict,
        "feedback_strengths": str(feedback.get("strengths")),
        "feedback_weaknesses": str(feedback.get("weaknesses")),
        "feedback_risk_factors": str(feedback.get("risk_factors")),
        "evaluation_parse_status": "ok" if not parse_error else parse_error,
        "raw_evaluation_output": raw_eval,
    }
    response_evaluations.append(evaluation_entry)

response_eval_df = pd.DataFrame(response_evaluations)

print(f"Parsed summary evaluations: {len(summary_eval_df)}, response evaluations: {len(response_eval_df)}")

Parsed summary evaluations: 30, response evaluations: 30


In [66]:
display(summary_eval_df.head())
summary_eval_df.to_csv("C:\\csagents\\agenticAIFoundations\\Agentic-AI-Foundation\\summary_evaluation.csv", index=False)   

,technical_accuracy,completeness,conciseness,hallucination_check,summary_overall_verdict,evaluation_parse_status,raw_evaluation_output
0,1.0,0.5,1,1,The summary accurately captures the key issue ...,ok,"```json\n{\n ""scores"": {\n ""technical_accu..."
1,1.0,1.0,1,1,The summary accurately captures all the key is...,ok,"```json\n{\n ""scores"": {\n ""technical_accu..."
2,1.0,1.0,1,1,The summary accurately reflects the original t...,ok,"```json\n{\n ""scores"": {\n ""technical_accu..."
3,1.0,0.8,1,1,The summary is mostly accurate but lacks menti...,ok,"```json\n{\n ""scores"": {\n ""technical_accu..."
4,1.0,1.0,1,1,The summary accurately and concisely reflects ...,ok,"{\n ""scores"": {\n ""technical_accuracy"": 1,..."


In [67]:
display(response_eval_df.head())
response_eval_df.to_csv( "C:\\csagents\\agenticAIFoundations\\Agentic-AI-Foundation\\response_evaluation.csv", index=False)   

,alignment_with_summary,actionability,tone_empathy,policy_compliance,response_overall_verdict,feedback_strengths,feedback_weaknesses,feedback_risk_factors,evaluation_parse_status,raw_evaluation_output
0,1,1,1,1,The generated response effectively addresses t...,The response directly addresses the customer's...,The response could be improved by providing a ...,There are minimal risk factors in the response...,ok,"{\n ""scores"": {\n ""alignment_with_summary""..."
1,1,1,1,1,The response is well-aligned with the ticket s...,The response directly addresses the customer's...,There are no weaknesses in the response as it ...,There is a potential risk if the policy regard...,ok,"{\n ""scores"": {\n ""alignment_with_summary""..."
2,1,1,1,1,The response is appropriate and effective as i...,The response aligns well with the ticket summa...,The response could offer a suggestion of commo...,There are no significant risk factors; however...,ok,"```json\n{\n ""scores"": {\n ""alignment_with..."
3,1,1,1,1,The response is well-aligned with the ticket s...,The response directly addresses the customer's...,The response could improve by mentioning any s...,"If the reversal does not process as promised, ...",ok,"```json\n{\n ""scores"": {\n ""alignment_with..."
4,1,1,1,1,The response is well-aligned with the ticket s...,The response is empathetic and acknowledges th...,No specific policy details are provided upfron...,Delayed response or processing after receiving...,ok,"```json\n{\n ""scores"": {\n ""alignment_with..."


# **Business Insights & Recommendations**

## **Business Insights**

The analysis shows that an agentic support workflow can convert unstructured customer complaints into a concise summary, generate a customer-facing response, and then evaluate that response against explicit quality criteria. The strongest business value comes from consistency: responses can be checked for issue alignment, actionability, empathy, and policy compliance before they reach a customer.

The evaluation output also highlights operational risks. A response may sound helpful while still making an unsupported promise, omitting contact details, or recommending a resolution that requires a policy check. Blank or unparseable evaluation rows should be treated as a monitoring signal rather than as successful results.

## **Recommendations**

1. Use the summarizer as the first step in support triage, but require the summary to preserve order details, customer intent, and the requested resolution.

2. Keep the evaluator as a separate quality-control step. Require valid structured JSON, log model and parameter settings, and route low-confidence, failed, or policy-sensitive cases to a human agent.

3. Add business rules before sending responses: verify replacement, refund, credit, and escalation eligibility instead of allowing the language model to claim that an action has already been completed.

4. Track metrics such as evaluation pass rate, JSON parsing failure rate, escalation rate, resolution time, and customer satisfaction. Review these metrics by issue type and model so the workflow can be improved using evidence.

5. Use the lower-cost model for routine evaluation and reserve the stronger model or human review for complex complaints, policy exceptions, and high-risk customers.

<font size=6>Power Ahead!</font>
___